# Module 5 lab: constraints in disposable SQLite

The database lives only in memory; SQL values are parameterized.

## Objectives and predictions

Predict 1: duplicate learner/course pair? Predict 2: missing course reference? Predict 3: status paused?

In [ ]:
import sqlite3
conn=sqlite3.connect(":memory:"); conn.execute("PRAGMA foreign_keys=ON")
conn.executescript("CREATE TABLE learners(id INTEGER PRIMARY KEY,name TEXT NOT NULL); CREATE TABLE courses(id INTEGER PRIMARY KEY,code TEXT NOT NULL UNIQUE,title TEXT NOT NULL); CREATE TABLE enrollments(learner_id INTEGER NOT NULL REFERENCES learners(id) ON DELETE CASCADE,course_id INTEGER NOT NULL REFERENCES courses(id) ON DELETE RESTRICT,status TEXT NOT NULL CHECK(status IN ('active','completed','cancelled')),PRIMARY KEY(learner_id,course_id));")
conn.execute("INSERT INTO learners(name) VALUES (?)",("Ana",)); conn.executemany("INSERT INTO courses(code,title) VALUES (?,?)",[("PY","Python"),("SQL","SQL")]); conn.execute("INSERT INTO enrollments VALUES (?,?,?)",(1,1,"active")); conn.commit()
def rejected(sql,p):
 try: conn.execute(sql,p); conn.commit(); return False
 except sqlite3.IntegrityError: conn.rollback(); return True
assert rejected("INSERT INTO enrollments VALUES (?,?,?)",(1,1,"active")); assert rejected("INSERT INTO enrollments VALUES (?,?,?)",(1,99,"active")); assert rejected("INSERT INTO enrollments VALUES (?,?,?)",(1,2,"paused"))

## Prediction answers

1. A duplicate pair violates the composite primary key. 2. A missing course violates the foreign key. 3. Paused violates the CHECK constraint. These are database-level protections, not merely application branches.

## Joins, aggregates, and lifecycle

A relationship table makes many-to-many explicit. Use placeholders rather than formatting values into SQL.

In [ ]:
rows=conn.execute("SELECT l.name,c.code,e.status FROM enrollments e JOIN learners l ON l.id=e.learner_id JOIN courses c ON c.id=e.course_id WHERE l.id=? ORDER BY c.code",(1,)).fetchall()
assert rows==[("Ana","PY","active")]
assert conn.execute("SELECT COUNT(*) FROM enrollments WHERE course_id=?",(1,)).fetchone()[0]==1
conn.execute("UPDATE enrollments SET status=? WHERE learner_id=? AND course_id=? AND status=?",("completed",1,1,"active")); conn.commit()
assert conn.execute("SELECT status FROM enrollments").fetchone()[0]=="completed"
try: conn.execute("DELETE FROM courses WHERE id=?",(1,)); conn.commit(); raise AssertionError("restrict")
except sqlite3.IntegrityError: conn.rollback()
conn.execute("DELETE FROM learners WHERE id=?",(1,)); conn.commit(); assert conn.execute("SELECT COUNT(*) FROM enrollments").fetchone()[0]==0

## AI-style review and TODO

A draft with no foreign keys and string-built SQL is unsafe. Guided TODO: write a parameterized learner/course join ordered by code.

In [ ]:
ai="SELECT * FROM enrollments WHERE learner_id=" + " + user_input"; assert "+" in ai
conn.execute("INSERT INTO learners(id,name) VALUES (?,?)",(2,"Ben")); conn.execute("INSERT INTO enrollments VALUES (?,?,?)",(2,2,"active")); conn.commit()
codes=[r[0] for r in conn.execute("SELECT c.code FROM enrollments e JOIN courses c ON c.id=e.course_id WHERE e.learner_id=? ORDER BY c.code",(2,))]
assert codes==["SQL"]
conn.execute("CREATE TABLE sections(id INTEGER PRIMARY KEY,course_id INTEGER NOT NULL REFERENCES courses(id) ON DELETE CASCADE,name TEXT NOT NULL,UNIQUE(course_id,name))")
conn.execute("INSERT INTO sections(course_id,name) VALUES (?,?)",(2,"Basics")); conn.commit(); assert rejected("INSERT INTO sections(course_id,name) VALUES (?,?)",(2,"Basics"))

## Independent challenge

Decide cascade or restrict for sections and test it. Exit answers: the composite key prevents duplicates; a primary key identifies a local row while a foreign key references another; restrict can protect history; parameters keep values as data; duplicate counts can drift without a source of truth.

Evidence: invariant/ER sketch, rejected writes, query results, delete behavior, rollback/migration notes, AI review, clean run.

## Baseline reproduction: slow path
Before constraints, duplicate and orphan rows are possible. Our schema now rejects them; record the attempted invalid writes as the baseline defect class.

In [ ]:
baseline_defects=["duplicate enrollment accepted","orphan enrollment accepted","invalid status accepted"]
assert len(baseline_defects)==3
print("Baseline defect classes:",baseline_defects)

## Pre-edit hypothesis
Write: “If the relationship table uses a composite key plus foreign keys and CHECK, then duplicate membership, missing parents, and invalid lifecycle values fail independently of application code.”

In [ ]:
pre_edit_hypothesis="composite key and constraints encode membership/reference/status invariants"
assert "constraints" in pre_edit_hypothesis

## Incremental guided implementation: cardinality
One learner-to-many enrollments and one course-to-many enrollments form a many-to-many relationship. One row in enrollments represents one pair; that is why the pair is the key.

In [ ]:
learner_courses=[(2,2)]
assert len(set(learner_courses))==1
assert conn.execute("SELECT COUNT(*) FROM enrollments WHERE learner_id=?",(2,)).fetchone()[0]==1

In [ ]:
def reference_courses(learner_id):
 return [row[0] for row in conn.execute("SELECT c.code FROM enrollments e JOIN courses c ON c.id=e.course_id WHERE e.learner_id=? ORDER BY c.code",(learner_id,))]
assert reference_courses(2)==["SQL"]

## Positive, negative, and failure checks
The positive join returns the intended row. Constraint failures are negative checks. Rollback is a failure-path check: an unfinished transaction must not leave a changed state.

In [ ]:
assert reference_courses(2)==["SQL"]
assert rejected("INSERT INTO enrollments VALUES (?,?,?)",(2,2,"active"))
conn.execute("BEGIN"); conn.execute("UPDATE enrollments SET status='cancelled' WHERE learner_id=?",(2,)); conn.rollback()
assert conn.execute("SELECT status FROM enrollments WHERE learner_id=?",(2,)).fetchone()[0]=="active"

## AI-style/broken-code critique
An AI schema that repeats course columns or uses a comma-separated list hides cardinality and cannot enforce uniqueness. A string-built SQL query is injection-prone.

In [ ]:
broken_schema="learner.course_1, learner.course_2; sql = '...'+user_value"
assert "course_1" in broken_schema and "+user_value" in broken_schema
print("Reject fixed repeated relationships and SQL concatenation.")

## Guided TODO: attempt
Write a parameterized lifecycle transition that changes only an active enrollment to completed and reports the affected row count.

In [ ]:
def todo_complete(learner_id,course_id):
 cur=conn.execute("UPDATE enrollments SET status=? WHERE learner_id=? AND course_id=? AND status=?",("completed",learner_id,course_id,"active")); conn.commit(); return cur.rowcount
# learner 2 currently active after the rollback check
assert todo_complete(2,2)==1 and conn.execute("SELECT status FROM enrollments WHERE learner_id=2").fetchone()[0]=="completed"

## Reference solution
The WHERE clause expresses the allowed transition; a zero row count is a conflict/absence result, not a success.

In [ ]:
def reference_complete(learner_id,course_id):
 cur=conn.execute("UPDATE enrollments SET status=? WHERE learner_id=? AND course_id=? AND status=?",("completed",learner_id,course_id,"active")); conn.commit(); return {"changed":cur.rowcount}
assert reference_complete(2,2)["changed"]==0

## Independent challenge
Add a status history table, make the current status source of truth explicit, and query the latest transition. State whether history or current status wins during repair.

In [ ]:
conn.execute("CREATE TABLE IF NOT EXISTS enrollment_history(learner_id INTEGER,course_id INTEGER,status TEXT,changed_at TEXT)")
conn.execute("INSERT INTO enrollment_history VALUES (2,2,?,?)",("completed","fixed-clock")); conn.commit()
assert conn.execute("SELECT status FROM enrollment_history WHERE learner_id=2").fetchone()[0]=="completed"

## Exit questions and Answers
Cardinality describes how many related rows exist; a many-to-many relationship needs a table. Primary keys identify local rows; foreign keys enforce references. RESTRICT protects meaningful history; CASCADE removes dependents intentionally. Parameters keep values as data. Constraints protect paths that bypass the API.

## Evidence handoff
Hand off invariants/ER sketch, baseline defect classes, hypothesis, cardinality reasoning, join/aggregate/lifecycle results, rejected writes, rollback, AI review, TODO/reference, history challenge, and clean execution.